# 03 - Analisis de jugadores

Identificacion de las carencias del Mallorca a nivel individual, comparando cada jugador
con los de su **mismo puesto** en LaLiga 2025/26.

Parte de los ficheros de `data/` generados en los notebooks 01 y 02. Se ejecuta de arriba
abajo desde cero.

**Ficheros que genera**

| Fichero | Contenido |
|---|---|
| `fbref_misc_con_posiciones.csv` | Estadisticas de FBref con la posicion propia asignada |
| `jugadores_eventos_laliga_2526.csv` | Acciones defensivas por jugador desde los eventos |
| `duelos_laliga_2526.csv` | Duelos y entradas con su resultado |
| `jugadores_eventos_ofensivo_laliga_2526.csv` | Metricas ofensivas desde los eventos |
| `metricas_jugadores_laliga_2526.csv` | **Fichero maestro** que usan los notebooks 04, 05 y 06 |

## 1. Carga y funcion de normalizacion de nombres

In [1]:
# Subimos a la carpeta raiz del proyecto para que las rutas relativas funcionen
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Importamos pandas para manejar los datos
import pandas as pd
# Importamos unidecode para normalizar nombres
from unidecode import unidecode

# Definimos la funcion que normaliza un nombre para cruzar entre fuentes
def normalizar(nombre):
    # Quitamos acentos, pasamos a minusculas y limpiamos separadores
    n = unidecode(str(nombre)).lower().replace(".", "").replace("-", " ")
    # Eliminamos espacios sobrantes
    return " ".join(n.split())

# Cargamos las estadisticas miscelaneas de FBref con su indice de cuatro niveles
fb_misc = pd.read_csv("data/fbref_misc_laliga_2526.csv", header=[0, 1], index_col=[0, 1, 2, 3])

# Comprobamos las dimensiones
print("FBref misc:", fb_misc.shape)

FBref misc: (611, 17)


## 2. Preparacion de la tabla de jugadores

Filtramos a jugadores con al menos 10 partidos completos (`90s`) para que las metricas
por 90 minutos tengan muestra suficiente.

In [2]:
# Partimos de la tabla original descargada de FBref
misc = fb_misc.copy()

# Aplanamos el MultiIndex quedandonos con el segundo nivel solo si es real
misc.columns = [c[0] if (not c[1] or str(c[1]).startswith("Unnamed")) else c[1]
                for c in misc.columns]

# Reseteamos el indice para tener liga, temporada, equipo y jugador como columnas
misc = misc.reset_index()

# Nos quedamos solo con jugadores con suficiente muestra
misc = misc[misc["90s"] >= 10].copy()

# Calculamos las metricas defensivas por 90 minutos
misc["int_90"] = misc["Int"] / misc["90s"]
misc["tklw_90"] = misc["TklW"] / misc["90s"]
misc["fls_90"] = misc["Fls"] / misc["90s"]
misc["crs_90"] = misc["Crs"] / misc["90s"]

# Creamos la clave de jugador normalizada
misc["clave"] = misc["player"].apply(normalizar)

# Creamos la clave de equipo normalizada
misc["clave_equipo"] = misc["team"].apply(normalizar)

# Comprobamos cuantos jugadores quedan
print("Jugadores con 10+ partidos completos:", len(misc))

Jugadores con 10+ partidos completos: 347


## 3. Asignacion de posiciones detalladas

**Limitacion de las fuentes.** Ni FBref ni Understat distinguen central de lateral ni
pivote de extremo: FBref agrupa en `DF`/`MF`/`FW`/`GK` y Understat en `D`/`M`/`F`.
Comparar a un central con un lateral en intercepciones no mide lo mismo.

Se construyo `posiciones_laliga_2526.csv`, fichero propio con las posiciones de los
**637 jugadores de campo** de los 20 equipos, tomadas de las fichas de plantilla de Marca
(`marca.com/resultados/futbol/<equipo>/plantilla/2025/`), con ajustes manuales en los
casos en que la ficha no correspondia al rol real (Pedri clasificado como extremo,
Mingueza y Juanlu Sanchez como centrales de ficha que jugaron de lateral, etc.).

*Los porteros se excluyen del fichero: FBref ya los identifica correctamente como `GK`.*

In [3]:
# Cargamos el fichero de posiciones de elaboracion propia
posiciones = pd.read_csv("data/posiciones_laliga_2526.csv")

# Normalizamos el nombre del jugador
posiciones["clave"] = posiciones["jugador"].apply(normalizar)

# Normalizamos el nombre del equipo
posiciones["clave_equipo"] = posiciones["equipo"].apply(normalizar)

# Corregimos el nombre del Oviedo para que coincida con el de FBref
posiciones["clave_equipo"] = posiciones["clave_equipo"].replace("real oviedo", "oviedo")

# Mostramos el reparto por posicion
print(posiciones["posicion"].value_counts())

posicion
central        113
lateral        105
delantero       99
extremo         97
mediocentro     82
sin_definir     67
mediapunta      41
pivote          33
Name: count, dtype: int64


### Equivalencias de nombres entre fuentes

FBref usa a veces el nombre completo (`Daniel Parejo`, `Isaac Palazon Camacho`) y Marca
el apodo corto (`Dani Parejo`, `Isi Palazon`). Sin resolverlo, 72 jugadores quedaban
sin emparejar. El cruce se hace por **equipo + jugador**, no solo por nombre, porque hay
homonimos en distintos clubes (dos David Lopez, dos Alvaro Garcia, dos Dani Rodriguez).

In [4]:
# Diccionario de equivalencias entre los nombres de FBref y los de Marca
equivalencias = {
    "iglesias": "juan iglesias", "catena": "alejandro catena",
    "jonny castro": "jonny", "viktor tsyhankov": "viktor tsygankov",
    "urko gonzalez": "urko gonzalez de zarate", "cucho": "cucho hernandez",
    "djene": "djene dakonam", "daniel vivian": "dani vivian",
    "carmona": "jose angel carmona", "jose luis gaya": "jose gaya",
    "isaac palazon camacho": "isi palazon", "javier guerra": "javi guerra",
    "olasagasti": "jon olasagasti", "nicolas gonzalez": "nico gonzalez",
    "daniel parejo": "dani parejo", "youssef lekhedim": "youssef enriquez",
    "roberto navarro": "robert navarro", "castrin": "andres castrin",
    "francisco perez": "fran perez", "alexandre alemao": "alemao",
    "raul": "raul garcia", "brugui": "roger brugue",
    "peque": "peque fernandez", "jofre": "jofre carreras",
    "kike": "kike garcia", "rahim bonkano": "rahim alhassane",
    "jose luis garcia vaya": "pepelu", "joaquin martinez gauna": "oso",
    "alfonso espino": "pacha espino"}

# Aplicamos las equivalencias sobre la clave de FBref
misc["clave"] = misc["clave"].replace(equivalencias)

### Jugadores no cubiertos por la ficha de Marca

Las fichas de plantilla reflejan la composicion actual del equipo, no la de toda la
temporada. Cuatro jugadores con minutos suficientes llegaron en el mercado de invierno
y no aparecian en ninguna plantilla. Otros tres figuraban en categoria generica
("Defensa", "Centrocampista") sin concretar.

*Limitacion: estas siete asignaciones son de elaboracion propia, no proceden de la fuente.*

In [5]:
# Posiciones de los jugadores traspasados en el mercado invernal
fichajes_invierno = pd.DataFrame([
    {"clave_equipo": "alaves", "clave": "carlos vicente", "posicion": "extremo"},
    {"clave_equipo": "celta vigo", "clave": "bryan zaragoza", "posicion": "extremo"},
    {"clave_equipo": "oviedo", "clave": "salomon rondon", "posicion": "delantero"},
    {"clave_equipo": "elche", "clave": "alvaro nunez", "posicion": "lateral"}])

# Posiciones de jugadores que Marca dejaba en categoria generica
genericos = pd.DataFrame([
    {"clave_equipo": "celta vigo", "clave": "miguel roman", "posicion": "mediocentro"},
    {"clave_equipo": "levante", "clave": "victor garcia", "posicion": "extremo"},
    {"clave_equipo": "valencia", "clave": "filip ugrinic", "posicion": "mediocentro"}])

# Unimos todas las fuentes de posiciones en una sola tabla
tabla_posiciones = pd.concat([posiciones[["clave_equipo", "clave", "posicion"]],
                              fichajes_invierno, genericos], ignore_index=True)

# Eliminamos las entradas genericas que ya han sido sustituidas
tabla_posiciones = tabla_posiciones[tabla_posiciones["posicion"] != "sin_definir"]

# Eliminamos duplicados quedandonos con la ultima asignacion
tabla_posiciones = tabla_posiciones.drop_duplicates(subset=["clave_equipo", "clave"], keep="last")

# Cruzamos la tabla de FBref con las posiciones asignadas
misc = misc.merge(tabla_posiciones, on=["clave_equipo", "clave"], how="left")

# Comprobamos que no queda ningun jugador de campo sin posicion
pendientes = misc[misc["posicion"].isna() & (misc["pos"] != "GK")]
print("Jugadores de campo sin posicion:", len(pendientes))

# Mostramos los pendientes si los hubiera
if len(pendientes) > 0:
    print(pendientes[["team", "player", "pos", "90s"]].to_string(index=False))

# Guardamos la tabla consolidada
misc.to_csv("data/fbref_misc_con_posiciones.csv", index=False)

# Mostramos el reparto definitivo por posicion
print("\n", misc["posicion"].value_counts())

Jugadores de campo sin posicion: 0



 posicion
central        64
lateral        63
mediocentro    51
extremo        50
delantero      45
pivote         27
mediapunta     24
Name: count, dtype: int64


## 4. Validacion de las posiciones con una segunda fuente

Las fichas de partido de WhoScored registran la posicion **en la que jugo realmente**
cada titular. Comparamos ambas fuentes para medir la fiabilidad de la asignacion.

WhoScored solo asigna posicion a los titulares: los suplentes aparecen como `Sub`.

In [6]:
# Subimos a la carpeta raiz del proyecto para que las rutas relativas funcionen
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd

posiciones = pd.read_csv("data/posiciones_laliga_2526.csv")
alineaciones = pd.read_csv("data/alineaciones_laliga_2526.csv")

print("POSICIONES")
print("  columnas:", posiciones.columns.tolist())
print(posiciones.head(3).to_string())

print("\nALINEACIONES")
print("  columnas:", alineaciones.columns.tolist())
print(alineaciones.head(3).to_string())

POSICIONES


  columnas: ['equipo', 'jugador', 'posicion']
     equipo          jugador posicion
0  Mallorca      Mateu Morey  lateral
1  Mallorca        Toni Lato  lateral
2  Mallorca  Marash Kumbulla  central

ALINEACIONES
  columnas: ['partido', 'team', 'player_id', 'player', 'posicion_ws', 'titular', 'edad', 'altura']
        partido         team  player_id           player posicion_ws  titular  edad  altura
0  1913882.json  Real Oviedo     141073  Aarón Escandell          GK     True    30     185
1  1913882.json  Real Oviedo     448447      Oier Luengo          DC     True    28     185
2  1913882.json  Real Oviedo     300918       Dani Calvo          DC     True    32     190


In [7]:
# Subimos a la carpeta raiz del proyecto para que las rutas relativas funcionen
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd
from unidecode import unidecode

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)

posiciones = pd.read_csv("data/posiciones_laliga_2526.csv")
alineaciones = pd.read_csv("data/alineaciones_laliga_2526.csv")


# Unificamos la forma de escribir los nombres antes de cruzar las dos fuentes
def normalizar(nombre):
    n = unidecode(str(nombre)).lower().replace(".", "").replace("-", " ")
    return " ".join(n.split())


posiciones["clave"] = posiciones["jugador"].apply(normalizar)

# Nos quedamos solo con las alineaciones titulares, porque los suplentes
# no llevan posicion real asignada
titulares = alineaciones[alineaciones["titular"] == True].copy()
titulares["clave"] = titulares["player"].apply(normalizar)

# Posicion mas repetida que WhoScored asigna a cada jugador
ws = (titulares
      .groupby("clave")["posicion_ws"]
      .agg(lambda s: s.mode().iat[0])
      .reset_index())

comparacion = posiciones.merge(ws, on="clave", how="inner")

print("jugadores en el fichero propio:", len(posiciones))
print("jugadores cruzados con WhoScored:", len(comparacion))

print("\nCATEGORIAS DE MI FICHERO")
print(posiciones["posicion"].value_counts())

print("\nCODIGOS DE WHOSCORED")
print(ws["posicion_ws"].value_counts())

print("\nTABLA CRUZADA")
print(pd.crosstab(comparacion["posicion"], comparacion["posicion_ws"]))

jugadores en el fichero propio: 637
jugadores cruzados con WhoScored: 464

CATEGORIAS DE MI FICHERO
posicion
central        113
lateral        105
delantero       99
extremo         97
mediocentro     82
sin_definir     67
mediapunta      41
pivote          33
Name: count, dtype: int64

CODIGOS DE WHOSCORED
posicion_ws
DC     103
FW      73
MC      63
DR      46
DL      40
GK      39
DMC     35
AMC     29
AML     25
ML      23
AMR     23
MR      14
DMR      5
FWR      4
DML      3
FWL      3
Name: count, dtype: int64

TABLA CRUZADA
posicion_ws  AMC  AML  AMR  DC  DL  DMC  DML  DMR  DR  FW  FWL  FWR  MC  ML  MR
posicion                                                                       
central        0    0    0  81   2    1    0    0   4   0    0    0   0   0   0
delantero      0    3    1   0   0    0    1    0   0  52    0    2   0   5   0
extremo        2   17   17   0   1    0    0    2   3  10    1    0   3  11   8
lateral        0    0    0  14  33    0    2    3  36   0    0

In [8]:
# Equivalencia entre los codigos de WhoScored y las categorias de mi fichero
EQUIVALENCIA = {
    "DC": "central",
    "DL": "lateral", "DR": "lateral", "DML": "lateral", "DMR": "lateral",
    "DMC": "pivote",
    "MC": "mediocentro",
    "AMC": "mediapunta",
    "AML": "extremo", "AMR": "extremo", "ML": "extremo", "MR": "extremo",
    "FW": "delantero", "FWL": "delantero", "FWR": "delantero",
}

# Agrupacion por lineas, siguiendo la propia taxonomia de WhoScored
LINEA_WS = {
    "DC": "defensa", "DL": "defensa", "DR": "defensa",
    "DML": "defensa", "DMR": "defensa",
    "DMC": "centrocampista", "MC": "centrocampista", "AMC": "centrocampista",
    "ML": "centrocampista", "MR": "centrocampista",
    "AML": "ataque", "AMR": "ataque",
    "FW": "ataque", "FWL": "ataque", "FWR": "ataque",
}

LINEA_PROPIA = {
    "central": "defensa", "lateral": "defensa",
    "pivote": "centrocampista", "mediocentro": "centrocampista",
    "mediapunta": "centrocampista",
    "extremo": "ataque", "delantero": "ataque",
}

# Dejamos fuera a los jugadores sin posicion asignada
val = comparacion[comparacion["posicion"] != "sin_definir"].copy()

# Traducimos el codigo de WhoScored a mi nomenclatura
val["ws_traducido"] = val["posicion_ws"].map(EQUIVALENCIA)
val["linea_ws"] = val["posicion_ws"].map(LINEA_WS)
val["linea_propia"] = val["posicion"].map(LINEA_PROPIA)

# Calculamos los dos niveles de coincidencia
exacta = (val["posicion"] == val["ws_traducido"]).mean() * 100
por_linea = (val["linea_propia"] == val["linea_ws"]).mean() * 100

print("jugadores validados:", len(val))
print(f"coincidencia exacta: {exacta:.1f} %")
print(f"coincidencia por lineas: {por_linea:.1f} %")

# Vemos en que categorias falla mas, por si hay que comentarlo en la memoria
print("\nACIERTO POR CATEGORIA")
detalle = (val.assign(acierto=val["posicion"] == val["ws_traducido"])
           .groupby("posicion")["acierto"]
           .agg(["size", "sum", "mean"])
           .rename(columns={"size": "n", "sum": "aciertos", "mean": "tasa"}))
detalle["tasa"] = (detalle["tasa"] * 100).round(1)
print(detalle.sort_values("tasa"))

jugadores validados: 450
coincidencia exacta: 72.2 %
coincidencia por lineas: 87.6 %

ACIERTO POR CATEGORIA
              n  aciertos  tasa
posicion                       
mediapunta   37        14  37.8
mediocentro  67        33  49.3
pivote       31        16  51.6
extremo      75        53  70.7
lateral      88        74  84.1
delantero    64        54  84.4
central      88        81  92.0


**92,0 % de coincidencia en los centrales**, que es el puesto clave del trabajo: 81 de 88
comparados. Sobre los 450 jugadores validados la coincidencia exacta es del 72,2 %, y
sube al 87,6 % agrupando por lineas (defensa, centro del campo, ataque).

El desacuerdo se concentra en dos puntos:

1. **Pivote frente a mediocentro**: las fuentes no coinciden en donde esta la frontera
   (mediapunta 37,8 % de acierto, mediocentro 49,3 %, pivote 51,6 %). Por eso el analisis
   del centro del campo trata ambos puestos como un solo grupo.
2. **Laterales que jugaron de central**: aqui la etiqueta de Marca es la desfasada, no el
   dato de WhoScored.

Se mantienen las posiciones de Marca como fuente principal porque estan documentadas y
son estables durante toda la temporada. El 92,0 % de los centrales es la cifra que se
declara como limitacion en la memoria.

## 5. Acciones defensivas desde los eventos

In [9]:
# Cargamos el conjunto unificado de eventos
eventos = pd.read_parquet("data/eventos_laliga_2526.parquet")

# Normalizamos el equipo corrigiendo los nombres que no coinciden
eventos["clave_equipo"] = eventos["team"].apply(normalizar).replace(
    {"atletico": "atletico madrid", "deportivo alaves": "alaves", "real oviedo": "oviedo"})

# Normalizamos el jugador corrigiendo los nombres que no coinciden
eventos["clave"] = eventos["player"].apply(normalizar).replace(
    {"jonny": "jonny castro", "jon olasagasti": "olasagasti"})

# Definimos las acciones defensivas que nos interesan
acciones_def = ["Tackle", "Interception", "Clearance", "BlockedPass",
                "Aerial", "BallRecovery", "Challenge"]

# Filtramos las acciones defensivas descartando eventos sin jugador
defensa = eventos[eventos["type"].isin(acciones_def) & eventos["player"].notna()]

# Contamos cada tipo de accion por jugador y equipo
resumen_def = defensa.groupby(["clave_equipo", "clave", "type"]).size().unstack(fill_value=0).reset_index()

# Cruzamos con la tabla de jugadores que ya tiene minutos y posiciones
jug = misc.merge(resumen_def, on=["clave_equipo", "clave"], how="left")

# Descartamos porteros y jugadores sin posicion asignada
jug = jug[jug["posicion"].notna()].copy()

# Calculamos cada accion defensiva por 90 minutos
for accion in acciones_def:
    jug[accion + "_90"] = jug[accion] / jug["90s"]

# Guardamos la tabla consolidada
jug.to_csv("data/jugadores_eventos_laliga_2526.csv", index=False)

# Comprobamos que no falta ningun jugador de campo
print("Jugadores de campo sin eventos:", jug["Clearance"].isna().sum(), "de", len(jug))

# Mostramos el volumen de cada accion en la liga
print("\n", defensa["type"].value_counts())

Jugadores de campo sin eventos: 2 de 324

 type
BallRecovery    30540
Clearance       19476
Aerial          19468
Tackle          12574
Interception     6119
Challenge        5844
BlockedPass      5346
Name: count, dtype: int64


*Nota sobre `Challenge`: en la nomenclatura Opta registra los intentos fallidos de robo
en los que el jugador es superado. Nunca tiene resultado "Successful", por lo que solo
es interpretable como volumen (cuantas veces te superan), no como ratio de acierto.*

## 6. Percentiles defensivos por puesto

In [10]:
# Definimos las metricas defensivas a comparar
metricas = ["Interception_90", "Tackle_90", "Clearance_90", "BlockedPass_90",
            "Aerial_90", "BallRecovery_90"]

# Definimos las etiquetas legibles de cada metrica
etiquetas = ["Intercepciones", "Entradas", "Despejes", "Bloqueos",
             "Duelos aereos", "Recuperaciones"]

# Calculamos el percentil de cada jugador dentro de su puesto
for m in metricas:
    jug["pct_" + m] = jug.groupby("posicion")[m].rank(pct=True) * 100

# Filtramos a los jugadores del Mallorca
mallorca_jug = jug[jug["team"] == "Mallorca"].sort_values(["posicion", "90s"], ascending=[True, False])

# Mostramos los percentiles por puesto
print("PERCENTILES DEFENSIVOS POR PUESTO")
print(mallorca_jug[["player", "posicion", "90s"] + ["pct_" + m for m in metricas]].round(0).to_string(index=False))

# Mostramos las medianas de la liga por puesto
print("\nMEDIANAS DE LA LIGA POR PUESTO")
print(jug.groupby("posicion")[metricas].median().round(2))

# Mostramos el tamano de cada grupo de referencia
print("\nJugadores por puesto:")
print(jug["posicion"].value_counts())

PERCENTILES DEFENSIVOS POR PUESTO
         player    posicion  90s  pct_Interception_90  pct_Tackle_90  pct_Clearance_90  pct_BlockedPass_90  pct_Aerial_90  pct_BallRecovery_90
 Martin Valjent     central 37.0                 14.0           38.0              88.0                69.0           31.0                  6.0
 Antonio Raillo     central 23.0                 17.0           16.0              42.0                42.0           95.0                 12.0
   Vedat Muriqi   delantero 35.0                  9.0            9.0             100.0                 4.0           91.0                 36.0
   Mateo Joseph   delantero 19.0                 93.0           93.0              91.0                96.0           36.0                 82.0
    Jan Virgili     extremo 20.0                 36.0           10.0              24.0                64.0           68.0                 38.0
   Johan Mojica     lateral 30.0                 26.0           32.0              48.0                19.0  

*Nota sobre el tamano de muestra: el grupo de pivotes tiene solo 27 jugadores, asi que
cada posicion del ranking vale casi 4 puntos porcentuales. Los percentiles en ese grupo
son mas gruesos y no deben interpretarse con precision.*

## 7. Porcentaje de acierto en duelos

Los eventos de duelo y entrada registran tanto los ganados como los perdidos,
lo que permite calcular ratios de acierto y no solo volumen.

In [11]:
# Filtramos los duelos aereos, entradas y desbordes con su resultado
duelos = eventos[eventos["type"].isin(["Aerial", "Tackle", "Challenge"]) & eventos["player"].notna()]

# Contamos cada combinacion de accion y resultado por jugador
res_duelos = duelos.groupby(["clave_equipo", "clave", "type", "outcome_type"]).size().unstack(fill_value=0)

# Calculamos el porcentaje de acierto de cada accion
res_duelos["pct_exito"] = res_duelos["Successful"] / (res_duelos["Successful"] + res_duelos["Unsuccessful"]) * 100

# Reseteamos el indice para poder consultarlo
res_duelos = res_duelos.reset_index()

# Guardamos la tabla de duelos
res_duelos.to_csv("data/duelos_laliga_2526.csv", index=False)

# Mostramos los duelos de los centrales del Mallorca
print("DUELOS DE LOS CENTRALES DEL MALLORCA")
print(res_duelos[(res_duelos["clave_equipo"] == "mallorca") &
                 (res_duelos["clave"].isin(["martin valjent", "antonio raillo"]))].round(1).to_string(index=False))

# Calculamos la mediana de acierto de los centrales de la liga
centrales_claves = set(zip(jug[jug["posicion"] == "central"]["clave_equipo"],
                           jug[jug["posicion"] == "central"]["clave"]))
mask = res_duelos.apply(lambda r: (r["clave_equipo"], r["clave"]) in centrales_claves, axis=1)

# Mostramos la mediana por tipo de accion
print("\nMEDIANA DE ACIERTO DE LOS CENTRALES:")
print(res_duelos[mask].groupby("type")["pct_exito"].median().round(1))

DUELOS DE LOS CENTRALES DEL MALLORCA
clave_equipo          clave      type  Successful  Unsuccessful  pct_exito
    mallorca antonio raillo    Aerial          76            38       66.7
    mallorca antonio raillo Challenge           0             3        0.0
    mallorca antonio raillo    Tackle          13            12       52.0
    mallorca martin valjent    Aerial          58            40       59.2
    mallorca martin valjent Challenge           0            10        0.0
    mallorca martin valjent    Tackle          34            16       68.0

MEDIANA DE ACIERTO DE LOS CENTRALES:


type
Aerial       58.2
Challenge     0.0
Tackle       57.6
Name: pct_exito, dtype: float64


**Los dos centrales tienen perfiles distintos, y ninguno anticipa.**

- **Valjent**: defensor reactivo. Despejes percentil 89 y bloqueos 71, pero intercepta
  0,63 por 90 frente a 1,06 de mediana (-41%). Acierta el 68% de sus entradas
  (mediana 57,6%): entra poco pero bien.
- **Raillo**: dominador aereo. Gana el 66,7% de sus duelos aereos (mediana 58,2%) y
  esta por debajo de la mediana en todo lo demas.

*Ninguno de los dos es un mal jugador en terminos absolutos. El problema es que el
equipo concedia 51 disparos de mas en juego abierto por el pasillo central, y ninguno
de los dos cortaba la jugada antes de que llegara al area.*

## 8. Los cuatro centrales del Mallorca

Revisamos tambien a los suplentes, sin el filtro de 10 partidos completos, por si el
recambio estaba en la propia plantilla.

*Cautela: con pocos minutos los ratios por 90 se disparan facilmente, y los suplentes
entran en contextos distintos (partidos decididos, ultimos minutos). Sus valores no son
directamente comparables con los de un titular.*

In [12]:
# Partimos de la tabla original de FBref sin filtrar por minutos
todos = fb_misc.copy()

# Aplanamos el MultiIndex de columnas
todos.columns = [c[0] if (not c[1] or str(c[1]).startswith("Unnamed")) else c[1]
                 for c in todos.columns]

# Reseteamos el indice y nos quedamos con el Mallorca
todos = todos.reset_index()
todos = todos[todos["team"] == "Mallorca"].copy()

# Creamos las claves de cruce con las equivalencias de nombre
todos["clave"] = todos["player"].apply(normalizar).replace(equivalencias)
todos["clave_equipo"] = todos["team"].apply(normalizar)

# Anadimos las acciones defensivas de los eventos
todos = todos.merge(resumen_def, on=["clave_equipo", "clave"], how="left")

# Posicion mas repetida en WhoScored y numero de titularidades de cada jugador
alineaciones = pd.read_csv("data/alineaciones_laliga_2526.csv")
titulares_ws = alineaciones[alineaciones["titular"] == True].copy()
titulares_ws["clave"] = titulares_ws["player"].apply(normalizar)
titulares_ws["clave_equipo"] = titulares_ws["team"].apply(normalizar).replace(
    {"atletico": "atletico madrid", "deportivo alaves": "alaves", "real oviedo": "oviedo"})
modal = (titulares_ws.groupby(["clave_equipo", "clave"])["posicion_ws"]
         .agg(posicion_ws=lambda s: s.mode().iat[0], total="size")
         .reset_index())

# Anadimos la posicion de WhoScored y sus titularidades
todos = todos.merge(modal[["clave_equipo", "clave", "posicion_ws", "total"]],
                    on=["clave_equipo", "clave"], how="left")

# Definimos los centrales de la plantilla segun Marca
centrales_marca = ["martin valjent", "antonio raillo", "marash kumbulla", "david lopez"]

# Nos quedamos con los que son centrales para Marca o para WhoScored
cb_mall = todos[todos["clave"].isin(centrales_marca) | (todos["posicion_ws"] == "DC")].copy()

# Calculamos los minutos totales jugados
cb_mall["minutos"] = cb_mall["90s"] * 90

# Calculamos las acciones defensivas por 90 minutos
for accion in ["Interception", "Tackle", "Clearance", "BlockedPass", "Aerial"]:
    cb_mall[accion + "_90"] = cb_mall[accion] / cb_mall["90s"]

# Mostramos a los centrales ordenados por minutos
print("CENTRALES DEL MALLORCA (todos, sin filtro de minutos)")
print(cb_mall.sort_values("minutos", ascending=False)[
    ["player", "minutos", "total", "Interception_90", "Tackle_90",
     "Clearance_90", "BlockedPass_90", "Aerial_90"]].round(2).to_string(index=False))

# Mostramos la mediana de los centrales titulares de la liga
print("\nMEDIANA DE LOS CENTRALES DE LA LIGA (10+ partidos completos)")
print(jug[jug["posicion"] == "central"][
    ["Interception_90", "Tackle_90", "Clearance_90",
     "BlockedPass_90", "Aerial_90"]].median().round(2))

CENTRALES DEL MALLORCA (todos, sin filtro de minutos)
         player  minutos  total  Interception_90  Tackle_90  Clearance_90  BlockedPass_90  Aerial_90
 Martin Valjent   3312.0   37.0             0.62       1.36          6.74            0.57       2.66
 Antonio Raillo   2070.0   23.0             0.70       1.09          5.30            0.39       4.96
    David López    837.0    9.0             0.86       1.18          8.71            0.75       4.52
Marash Kumbulla    441.0    5.0             0.41       0.61          2.86            0.61       3.27

MEDIANA DE LOS CENTRALES DE LA LIGA (10+ partidos completos)
Interception_90    1.08
Tackle_90          1.62
Clearance_90       5.62
BlockedPass_90     0.46
Aerial_90          3.35
dtype: float64


**Hallazgo clave: los cuatro centrales del Mallorca estan por debajo de la mediana de
la liga en intercepciones** (0,87 / 0,70 / 0,63 / 0,41 frente a 1,06). No es un problema
de dos jugadores: en toda la plantilla no habia un central que anticipara, y por tanto
**no habia recambio interno**.

David Lopez (827 minutos) registra los mejores numeros de los cuatro, con margen para
haber jugado mas, pero su muestra es corta y el problema de fondo no se resolvia con eso.

## 9. Metricas ofensivas desde los eventos

Definiciones propias y simplificadas, a declarar en la memoria:
- **Pases progresivos**: completados que avanzan 25 o mas unidades hacia la porteria rival
- **Pases al ultimo tercio**: completados que empiezan fuera y acaban dentro del ultimo tercio
- **Pases al area**: completados que terminan dentro del area rival

In [13]:
# Filtramos los pases de toda la liga
pases = eventos[(eventos["type"] == "Pass") & eventos["player"].notna()].copy()

# Marcamos los pases completados
pases["completado"] = pases["outcome_type"] == "Successful"

# Marcamos los pases que entran en el ultimo tercio desde fuera de el
pases["ultimo_tercio"] = pases["completado"] & (pases["x"] < 66.7) & (pases["end_x"] >= 66.7)

# Marcamos los pases completados que acaban dentro del area rival
pases["al_area"] = pases["completado"] & (pases["end_x"] >= 83) & pases["end_y"].between(21.1, 78.9)

# Marcamos los pases progresivos
pases["progresivo"] = pases["completado"] & ((pases["end_x"] - pases["x"]) >= 25)

# Agrupamos los pases por jugador y equipo
of_pases = pases.groupby(["clave_equipo", "clave"]).agg(
    pases_intentados=("completado", "size"),
    pases_completados=("completado", "sum"),
    pases_ultimo_tercio=("ultimo_tercio", "sum"),
    pases_area=("al_area", "sum"),
    pases_progresivos=("progresivo", "sum")).reset_index()

# Contamos los regates completados por jugador
regates = eventos[(eventos["type"] == "TakeOn") & (eventos["outcome_type"] == "Successful")]
of_regates = regates.groupby(["clave_equipo", "clave"]).size().reset_index(name="regates")

# Contamos los tiros por jugador
tiros = eventos[eventos["is_shot"] == True]
of_tiros = tiros.groupby(["clave_equipo", "clave"]).size().reset_index(name="tiros")

# Unimos las metricas ofensivas a la tabla de jugadores
jug_of = jug.merge(of_pases, on=["clave_equipo", "clave"], how="left")
jug_of = jug_of.merge(of_regates, on=["clave_equipo", "clave"], how="left")
jug_of = jug_of.merge(of_tiros, on=["clave_equipo", "clave"], how="left")

# Rellenamos con cero a quien no tenga regates o tiros
jug_of[["regates", "tiros"]] = jug_of[["regates", "tiros"]].fillna(0)

# Calculamos el porcentaje de acierto en el pase
jug_of["pct_pase"] = jug_of["pases_completados"] / jug_of["pases_intentados"] * 100

# Definimos las metricas ofensivas
metricas_of = ["pases_completados", "pases_progresivos", "pases_ultimo_tercio",
               "pases_area", "regates", "tiros"]

# Calculamos cada metrica ofensiva por 90 minutos
for m in metricas_of:
    jug_of[m + "_90"] = jug_of[m] / jug_of["90s"]

# Guardamos la tabla con metricas defensivas y ofensivas
jug_of.to_csv("data/jugadores_eventos_ofensivo_laliga_2526.csv", index=False)

# Comprobamos las dimensiones
print("Tabla completa:", jug_of.shape)

Tabla completa: (324, 62)


## 10. El centro del campo: el caso Darder

Pivotes y mediocentros se analizan **como un solo grupo**: compiten por los mismos
minutos y las fuentes no coinciden en donde esta la frontera entre ambos puestos.

*Al incluir pivotes, cuya funcion principal es recuperar, los perfiles mas creativos
salen penalizados en las metricas defensivas. Es una decision consciente.*

In [14]:
# Definimos los puestos que compiten por los mismos minutos
puestos_medio = ["pivote", "mediocentro"]

# Filtramos a todos los centrocampistas de la liga
medio = jug_of[jug_of["posicion"].isin(puestos_medio) & (jug_of["90s"] >= 10)].copy()

# Calculamos el percentil de cada metrica dentro del grupo unificado
for m in metricas + [x + "_90" for x in metricas_of]:
    medio["pctl_" + m] = medio[m].rank(pct=True) * 100

# Filtramos a los centrocampistas del Mallorca
medio_mall = medio[medio["team"] == "Mallorca"].sort_values("90s", ascending=False)

# Mostramos los percentiles defensivos
print(f"CENTROCAMPISTAS DEL MALLORCA - PERCENTIL DEFENSIVO (grupo de {len(medio)})")
print(medio_mall[["player", "posicion", "90s"] + ["pctl_" + m for m in metricas]].round(0).to_string(index=False))

# Mostramos los percentiles ofensivos
print("\nPERCENTIL OFENSIVO")
print(medio_mall[["player"] + ["pctl_" + m + "_90" for m in metricas_of]].round(0).to_string(index=False))

CENTROCAMPISTAS DEL MALLORCA - PERCENTIL DEFENSIVO (grupo de 78)
         player    posicion  90s  pctl_Interception_90  pctl_Tackle_90  pctl_Clearance_90  pctl_BlockedPass_90  pctl_Aerial_90  pctl_BallRecovery_90
     Samu Costa      pivote 30.0                  49.0            65.0               94.0                 97.0            96.0                  23.0
   Sergi Darder mediocentro 30.0                  51.0             4.0               13.0                 30.0             4.0                  32.0
 Omar Mascarell      pivote 25.0                  84.0            52.0               95.0                 71.0            81.0                  43.0
  Manu Morlanes mediocentro 20.0                  65.0            92.0               39.0                 62.0            44.0                  68.0
Antonio Sánchez mediocentro 12.0                  69.0            38.0               75.0                 88.0            22.0                  90.0

PERCENTIL OFENSIVO
         player  pctl

## 11. Metricas avanzadas de Understat

In [15]:
# Cargamos las estadisticas de jugador de Understat
us = pd.read_csv("data/jugadores_laliga_2526.csv")

# Normalizamos las claves de cruce
us["clave"] = us["player"].apply(normalizar)
us["clave_equipo"] = us["team"].apply(normalizar).replace({"real oviedo": "oviedo"})

# Equivalencias entre los nombres de Understat y los de nuestra tabla
equiv_us = {"manuel morlanes": "manu morlanes",
            "pablo barrios rivas": "pablo barrios",
            "urko gonzalez": "urko gonzalez de zarate",
            "azz eddine ounahi": "azzedine ounahi",
            "jon ander olasagasti": "jon olasagasti",
            "javier guerra": "javi guerra",
            "daniel parejo": "dani parejo",
            "pape alassane gueye": "pape gueye",
            "santiago comesana": "santi comesana"}

# Aplicamos las equivalencias
us["clave"] = us["clave"].replace(equiv_us)

# Definimos las metricas ofensivas de Understat
metricas_us = ["goals", "assists", "xg", "xa", "key_passes", "xg_chain", "xg_buildup"]

# Calculamos cada metrica por 90 minutos
for m in metricas_us:
    us[m + "_90"] = us[m] / us["minutes"] * 90

# Unimos las metricas de Understat a los centrocampistas
medio_us = medio.merge(us[["clave_equipo", "clave", "minutes"] + [m + "_90" for m in metricas_us]]
                       .drop_duplicates(["clave_equipo", "clave"]),
                       on=["clave_equipo", "clave"], how="left")

# Nos quedamos con los que tienen datos de Understat
medio_us = medio_us[medio_us["xg_90"].notna()].copy()

# Calculamos el percentil de cada metrica dentro del grupo
for m in metricas_us:
    medio_us["pctl_" + m] = medio_us[m + "_90"].rank(pct=True) * 100

# Filtramos a los centrocampistas del Mallorca
mall_us = medio_us[medio_us["team"] == "Mallorca"].sort_values("90s", ascending=False)

# Mostramos los valores por 90 minutos
print("VALORES POR 90 MINUTOS (Understat)")
print(mall_us[["player", "minutes"] + [m + "_90" for m in metricas_us]].round(2).to_string(index=False))

# Mostramos los percentiles
print("\nPERCENTILES FRENTE A LA LIGA")
print(mall_us[["player"] + ["pctl_" + m for m in metricas_us]].round(0).to_string(index=False))

# Mostramos el tamano del grupo de referencia
print("\nGrupo de referencia:", len(medio_us), "centrocampistas")

VALORES POR 90 MINUTOS (Understat)
         player  minutes  goals_90  assists_90  xg_90  xa_90  key_passes_90  xg_chain_90  xg_buildup_90
     Samu Costa   2731.0      0.23        0.07   0.14   0.07           0.56         0.31           0.17
   Sergi Darder   2715.0      0.03        0.17   0.07   0.18           1.89         0.35           0.24
 Omar Mascarell   2251.0      0.08        0.08   0.01   0.02           0.32         0.25           0.22
  Manu Morlanes   1801.0      0.15        0.00   0.08   0.07           0.85         0.42           0.28
Antonio Sánchez   1016.0      0.00        0.00   0.05   0.09           1.06         0.35           0.22

PERCENTILES FRENTE A LA LIGA
         player  pctl_goals  pctl_assists  pctl_xg  pctl_xa  pctl_key_passes  pctl_xg_chain  pctl_xg_buildup
     Samu Costa        94.0          43.0     86.0     42.0             22.0           34.0             17.0
   Sergi Darder        34.0          73.0     56.0     81.0             91.0           42.0  

**Darder: especialista del ultimo pase, pero no de la construccion.**

A favor: pases al area 2,52 por 90 (percentil 90 dentro del grupo), pases clave 1,89
(percentil 91), xA 0,18 (percentil 81), regates 1,14 (percentil 82). Es el jugador que
mas ocasiones fabrica del centro del campo.

En contra: **percentil 0 en entradas ganadas** entre los 51 mediocentros de la liga, y
por debajo de la mediana en cinco de las seis metricas defensivas.

Y el dato que decide: **Morlanes le supera en xGBuildup** (0,28 frente a 0,24) y en
xGChain (0,42 frente a 0,35), participando mas en las posesiones que generan peligro,
con un 88,6% de acierto en el pase frente al 78,2% de Darder, y siendo de los que mas
roban (percentil 92 en entradas).

**Conclusion:** la titularidad de Darder no se sostiene solo por su aportacion ofensiva.
Morlanes aporta mas en conjunto y ademas defiende. Darder encaja mejor como recurso para
partidos en los que el equipo necesite fabricar ocasiones.

*Nota: goles y asistencias en bruto son muy ruidosos en centrocampistas. Samu Costa
marco 0,23 por 90 con 0,14 de xG, y Morlanes 0,15 con 0,08: sobrerendimiento que rara vez
se repite. Para decisiones deben usarse las metricas esperadas.*

*Fran Beltran (Girona) se excluye de esta comparacion: ficho en el mercado invernal y
cada fuente registra un tramo distinto de su temporada (FBref el Girona, Understat el Celta).*

## 12. Jugadores de banda

In [16]:
# Nos quedamos con laterales y extremos
banda = jug_of[jug_of["posicion"].isin(["lateral", "extremo"]) & (jug_of["90s"] >= 10)].copy()

# Definimos las metricas ofensivas de banda
of_banda = ["crs_90", "pases_area_90", "regates_90", "pases_ultimo_tercio_90", "tiros_90"]

# Definimos las metricas defensivas de banda
de_banda = ["Interception_90", "Tackle_90", "BallRecovery_90", "Aerial_90"]

# Calculamos el percentil de cada metrica dentro de cada puesto
for m in of_banda + de_banda:
    banda["pct_" + m] = banda.groupby("posicion")[m].rank(pct=True) * 100

# Construimos un indice ofensivo como media de los percentiles ofensivos
banda["indice_of"] = banda[["pct_" + m for m in of_banda]].mean(axis=1)

# Construimos un indice defensivo como media de los percentiles defensivos
banda["indice_de"] = banda[["pct_" + m for m in de_banda]].mean(axis=1)

# Filtramos a los jugadores de banda del Mallorca
banda_mall = banda[banda["team"] == "Mallorca"].sort_values(["posicion", "90s"], ascending=[True, False])

# Mostramos los valores por 90 minutos
print("JUGADORES DE BANDA DEL MALLORCA - VALORES POR 90 MINUTOS")
print(banda_mall[["player", "posicion", "90s"] + of_banda + de_banda].round(2).to_string(index=False))

# Mostramos la mediana de la liga por puesto
print("\nMEDIANA DE LA LIGA POR PUESTO")
print(banda.groupby("posicion")[of_banda + de_banda].median().round(2))

# Mostramos los indices resumidos
print("\nINDICES RESUMIDOS (media de percentiles)")
print(banda_mall[["player", "posicion", "indice_of", "indice_de"]].round(0).to_string(index=False))

JUGADORES DE BANDA DEL MALLORCA - VALORES POR 90 MINUTOS


      player posicion  90s  crs_90  pases_area_90  regates_90  pases_ultimo_tercio_90  tiros_90  Interception_90  Tackle_90  BallRecovery_90  Aerial_90
 Jan Virgili  extremo 19.9    2.36           1.21        2.46                    0.70      1.61             0.35       0.75             3.42       1.86
Johan Mojica  lateral 30.3    4.39           1.82        0.50                    4.49      0.56             0.79       1.68             5.87       1.25
Pablo Maffeo  lateral 29.6    2.87           1.69        1.15                    2.91      0.27             1.22       2.26             3.75       1.86

MEDIANA DE LA LIGA POR PUESTO
          crs_90  pases_area_90  regates_90  pases_ultimo_tercio_90  tiros_90  Interception_90  Tackle_90  BallRecovery_90  Aerial_90
posicion                                                                                                                             
extremo     2.71           1.72        1.24                    1.58      1.98             0.5

*Los indices son una simplificacion propia: media de los percentiles de cada bloque.
Dan el mismo peso a todas las metricas, lo que no siempre es correcto. Sirven para situar
a un jugador de un vistazo, no como medida definitiva.*

**Las bandas no eran el problema:**

- **Maffeo**: el mas completo del equipo. Por encima de la mediana en ambas facetas:
  regates 1,15 (mediana 0,55), intercepciones 1,22 (mediana 1,05), entradas 2,26 (mediana 1,87).
- **Mojica**: lateral muy ofensivo (centros 4,39 frente a 2,40 de mediana, +83%) que
  apenas disputa. Sus recuperaciones son 5,87 frente a 3,91 (+50%) pero intercepta poco:
  patron tipico de un equipo replegado, donde los rechaces caen en campo propio.
- **Virgili**: solo aporta el regate (2,46 frente a 1,24). En pases al ultimo tercio esta
  a menos de la mitad de la mediana (0,70 frente a 1,58).

**El agujero real no es de rendimiento sino de plantilla: un solo extremo llego a 10
partidos completos en toda la temporada.**

## 13. Inventario de fortalezas

In [17]:
# Definimos todas las metricas a revisar
met_todas = metricas + [m + "_90" for m in metricas_of]

# Calculamos el percentil de cada jugador dentro de su puesto
for m in met_todas:
    jug_of["p_" + m] = jug_of.groupby("posicion")[m].rank(pct=True) * 100

# Nos quedamos con los jugadores del Mallorca
mall = jug_of[jug_of["team"] == "Mallorca"]

# Buscamos todas las fortalezas: percentil 75 o superior
print("FORTALEZAS INDIVIDUALES (percentil 75 o mas en su puesto)")
for _, f in mall.iterrows():
    # Recogemos las metricas donde destaca
    fuertes = [(m.replace("_90", ""), round(f["p_" + m])) for m in met_todas if f["p_" + m] >= 75]
    # Mostramos al jugador si tiene alguna fortaleza
    if fuertes:
        print(f"\n  {f['player']} ({f['posicion']}, {f['90s']:.1f} partidos)")
        for nombre, pct in sorted(fuertes, key=lambda x: -x[1]):
            print(f"      {nombre:24} percentil {pct}")

FORTALEZAS INDIVIDUALES (percentil 75 o mas en su puesto)

  Antonio Raillo (central, 23.0 partidos)
      Aerial                   percentil 95

  Antonio Sánchez (mediocentro, 11.9 partidos)
      BlockedPass              percentil 92
      Clearance                percentil 88
      BallRecovery             percentil 88
      Interception             percentil 76

  Jan Virgili (extremo, 19.9 partidos)
      regates                  percentil 86

  Johan Mojica (lateral, 30.3 partidos)
      BallRecovery             percentil 98
      pases_progresivos        percentil 94
      pases_ultimo_tercio      percentil 84
      pases_area               percentil 77

  Manu Morlanes (mediocentro, 19.6 partidos)
      Tackle                   percentil 92
      pases_ultimo_tercio      percentil 82

  Martin Valjent (central, 36.8 partidos)
      Clearance                percentil 88

  Mateo Joseph (delantero, 18.9 partidos)
      BlockedPass              percentil 96
      Interception    


  Pablo Torre (mediapunta, 17.8 partidos)
      Clearance                percentil 92
      pases_ultimo_tercio      percentil 92
      pases_progresivos        percentil 79

  Samu Costa (pivote, 30.0 partidos)
      BlockedPass              percentil 100
      tiros                    percentil 93
      Aerial                   percentil 89
      Clearance                percentil 85

  Sergi Darder (mediocentro, 29.8 partidos)
      pases_area               percentil 90

  Vedat Muriqi (delantero, 34.8 partidos)
      Clearance                percentil 100
      Aerial                   percentil 91
      tiros                    percentil 89


**Cuidado al leer esta tabla:** `Clearance` aparece como fortaleza en nueve de los trece
jugadores, incluidos Muriqi (percentil 100) y Pablo Torre (92), que son delantero y
mediapunta. Que el delantero centro sea el que mas despeja de la liga en su puesto no es
una virtud: es la consecuencia del bloque bajo documentado en el notebook 02. Esas
metricas son sintomas del sistema, no cualidades individuales.

**Fortalezas reales, descontando ese efecto:**

- **Mojica**: recuperaciones percentil 98, pases progresivos 94, al ultimo tercio 84
- **Muriqi**: duelos aereos 91, tiros 89, 23 goles (2o maximo goleador de LaLiga)
- **Samu Costa**: bloqueos 100, tiros 93, duelos aereos 89
- **Darder**: pases al area 90
- **Morlanes**: entradas 92
- **Maffeo**: regates 84
- **Raillo**: duelos aereos 95

**Lectura de conjunto:** los mimbres individuales existian (un delantero de nivel, un
lateral con buena salida, un pivote solvente, un creador de ocasiones). El problema era
colectivo: estructura defensiva y capacidad de llegar al area rival.

## 14. La plantilla del Mallorca en bruto

Antes de construir el fichero maestro, una mirada directa a los registros de Understat
para los jugadores del Mallorca: minutos, goles, xG, asistencias y participacion en las
posesiones que acaban en remate.

In [18]:
jugadores = pd.read_csv("data/jugadores_laliga_2526.csv")
mallorca = jugadores[jugadores["team"] == "Mallorca"]
print(mallorca[["player", "position", "matches", "minutes", "goals", "xg",
                "np_goals", "np_xg", "assists", "xa", "shots", "key_passes",
                "xg_chain", "xg_buildup"]]
      .sort_values("minutes", ascending=False)
      .to_string(index=False))

             player position  matches  minutes  goals        xg  np_goals     np_xg  assists       xa  shots  key_passes  xg_chain  xg_buildup
     Martin Valjent        D       37     3311      0  0.967719         0  0.967719        0 0.140044     17           3  9.191905    8.853639
       Vedat Muriqi      F S       37     3150     23 21.630785        18 16.427842        1 2.485310    114          24 18.902258    4.643540
          Leo Román       GK       33     2970      0  0.000000         0  0.000000        0 0.000000      0           0  3.981423    3.981423
         Samú Costa      M S       34     2731      7  4.277460         7  4.277460        2 1.979984     39          17  9.480698    5.158837
       Johan Mojica      D S       35     2723      0  0.806186         0  0.806186        4 3.927842     17          35  9.051306    5.788200
       Sergi Darder      M S       38     2715      1  2.258102         1  2.258102        5 5.543174     36          57 10.621093    7.191009

## 15. Acciones defensivas del Mallorca por 90 minutos

Entradas, intercepciones, despejes, bloqueos de pase y duelos aereos de cada jugador del
Mallorca, normalizados por 90 minutos a partir de los eventos Opta.

In [19]:
# Subimos a la carpeta raiz del proyecto para que las rutas relativas funcionen
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd
from unidecode import unidecode

pd.set_option("display.width", 250)

eventos = pd.read_parquet("data/eventos_laliga_2526.parquet")
jugadores = pd.read_csv("data/jugadores_laliga_2526.csv")
posiciones = pd.read_csv("data/posiciones_laliga_2526.csv")


# Unificamos la forma de escribir los nombres entre las tres fuentes
def normalizar(nombre):
    n = unidecode(str(nombre)).lower().replace(".", "").replace("-", " ")
    return " ".join(n.split())


ACCIONES_DEF = ["Tackle", "Interception", "Clearance", "BlockedPass", "Aerial"]

# Contamos las acciones defensivas de cada jugador del Mallorca
acciones = eventos[(eventos["team"] == "Mallorca")
                   & (eventos["type"].isin(ACCIONES_DEF))].copy()
acciones["clave"] = acciones["player"].apply(normalizar)

defensa = (acciones.groupby(["clave", "type"])
           .size()
           .unstack(fill_value=0)
           .reset_index())

# Traemos los minutos de Understat cruzando por el nombre normalizado
minutos = jugadores[jugadores["team"] == "Mallorca"][["player", "minutes"]].copy()
minutos["clave"] = minutos["player"].apply(normalizar)

# Cuatro jugadores se escriben distinto en WhoScored y en Understat
CORRECCIONES_NOMBRE = {
    "antonio raillo": "raillo",
    "justin kalumba": "justin noel kalumba",
    "manu morlanes": "manuel morlanes",
    "zito luvumbo": "zito",
}

# Traemos tambien la posicion de mi fichero propio
pos = posiciones[posiciones["equipo"] == "Mallorca"][["jugador", "posicion"]].copy()
pos["clave"] = pos["jugador"].apply(normalizar)

# Aplicamos las mismas correcciones al fichero de posiciones
pos["clave"] = pos["clave"].replace(CORRECCIONES_NOMBRE)

defensa["clave"] = defensa["clave"].replace(CORRECCIONES_NOMBRE)

tabla = defensa.merge(minutos[["clave", "player", "minutes"]], on="clave", how="left")
tabla = tabla.merge(pos[["clave", "posicion"]], on="clave", how="left")

# Avisamos si algun jugador se queda sin minutos por no cruzar el nombre
sin_cruzar = tabla[tabla["minutes"].isna()]["clave"].tolist()
if sin_cruzar:
    print("sin cruzar con Understat:", sin_cruzar, "\n")

# Nos quedamos con los que tienen minutos suficientes para comparar
tabla = tabla[tabla["minutes"] >= 450].copy()

# Pasamos cada accion a valores por 90 minutos
for col in ACCIONES_DEF:
    if col in tabla.columns:
        tabla[col + "_90"] = (tabla[col] / tabla["minutes"] * 90).round(2)

columnas = (["player", "posicion", "minutes"]
            + [c + "_90" for c in ACCIONES_DEF if c + "_90" in tabla.columns])

print(tabla[columnas]
      .sort_values("minutes", ascending=False)
      .to_string(index=False))

         player    posicion  minutes  Tackle_90  Interception_90  Clearance_90  BlockedPass_90  Aerial_90
 Martin Valjent     central     3311       1.36             0.63          6.74            0.57       2.66
   Vedat Muriqi   delantero     3150       0.26             0.06          1.66            0.17       7.29
      Leo Román         NaN     2970       0.03             0.03          0.67            0.00       0.12
     Samú Costa      pivote     2731       2.14             0.86          3.10            1.52       3.63
   Johan Mojica     lateral     2723       1.69             0.79          3.17            0.50       1.26
   Sergi Darder mediocentro     2715       0.93             0.86          0.80            0.60       0.50
   Pablo Maffeo     lateral     2690       2.24             1.20          3.85            0.64       1.84
 Omar Mascarell      pivote     2251       1.92             1.40          3.52            0.96       2.68
         Raíllo     central     2069       1.0

Valjent, con 3.311 minutos, registra **0,63 intercepciones por 90**. Es el central que mas
juega y el que menos anticipa, y ese par de datos es el que acaba sosteniendo la propuesta
de refuerzo.

## 16. Referencias de la liga: los 65 centrales

Para saber si un registro es bueno o malo hace falta contra que compararlo. Se toman todos
los centrales de LaLiga con al menos 600 minutos y se calculan sus medianas y cuartiles.

Ese minimo de minutos deja **65 centrales** con las cinco metricas completas. Es el grupo
de referencia del trabajo, y de el salen los umbrales de la criba del notebook 04.

In [20]:
# Subimos a la carpeta raiz del proyecto para que las rutas relativas funcionen
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd
from unidecode import unidecode

pd.set_option("display.width", 250)

eventos = pd.read_parquet("data/eventos_laliga_2526.parquet")
jugadores = pd.read_csv("data/jugadores_laliga_2526.csv")
posiciones = pd.read_csv("data/posiciones_laliga_2526.csv")


# Unificamos la forma de escribir los nombres entre las tres fuentes
def normalizar(nombre):
    n = unidecode(str(nombre)).lower().replace(".", "").replace("-", " ")
    return " ".join(n.split())


# Nombres de jugador que cada fuente escribe de forma distinta
CORRECCIONES_NOMBRE = {
    "antonio raillo": "raillo",
    "justin kalumba": "justin noel kalumba",
    "manu morlanes": "manuel morlanes",
    "zito luvumbo": "zito",
}

# Nombres de equipo que cada fuente escribe de forma distinta
CORRECCIONES_EQUIPO = {
    "atletico": "atletico madrid",
    "deportivo alaves": "alaves",
    "real oviedo": "oviedo",
}

ACCIONES_DEF = ["Tackle", "Interception", "Clearance", "BlockedPass", "Aerial"]


# ------------------------------------------------------------------
# 1. Acciones defensivas de todos los jugadores de la competicion
# ------------------------------------------------------------------
todas = eventos[eventos["type"].isin(ACCIONES_DEF)].copy()
todas["clave"] = todas["player"].apply(normalizar).replace(CORRECCIONES_NOMBRE)

acciones_liga = (todas.groupby(["clave", "type"])
                 .size()
                 .unstack(fill_value=0)
                 .reset_index())


# ------------------------------------------------------------------
# 2. Minutos y posicion, cruzando por nombre Y equipo
# ------------------------------------------------------------------
min_liga = jugadores[["player", "team", "minutes"]].copy()
min_liga["clave"] = min_liga["player"].apply(normalizar).replace(CORRECCIONES_NOMBRE)
min_liga["equipo_clave"] = min_liga["team"].apply(normalizar).replace(CORRECCIONES_EQUIPO)

pos_liga = posiciones[["jugador", "equipo", "posicion"]].copy()
pos_liga["clave"] = pos_liga["jugador"].apply(normalizar).replace(CORRECCIONES_NOMBRE)
pos_liga["equipo_clave"] = pos_liga["equipo"].apply(normalizar).replace(CORRECCIONES_EQUIPO)

liga = (pos_liga
        .merge(min_liga[["clave", "equipo_clave", "minutes"]],
               on=["clave", "equipo_clave"], how="inner")
        .merge(acciones_liga, on="clave", how="inner"))

print("jugadores cruzados:", len(liga), "de", len(pos_liga))
print("duplicados por nombre:", int(liga["clave"].duplicated().sum()))

# Si bajan mucho los cruzados, el problema son los nombres de equipo
faltan = sorted(set(pos_liga["equipo_clave"]) - set(min_liga["equipo_clave"]))
if faltan:
    print("equipos que no cruzan:", faltan)


# ------------------------------------------------------------------
# 3. Referencias de la liga entre los centrales con minutos suficientes
# ------------------------------------------------------------------
centrales = liga[(liga["posicion"] == "central") & (liga["minutes"] >= 600)].copy()

for col in ACCIONES_DEF:
    if col in centrales.columns:
        centrales[col + "_90"] = (centrales[col] / centrales["minutes"] * 90).round(2)

columnas_90 = [c + "_90" for c in ACCIONES_DEF if c + "_90" in centrales.columns]

print("\ncentrales comparados:", len(centrales))
print("\nREFERENCIAS DE LA LIGA (centrales, minimo 600 minutos)")

for col in ACCIONES_DEF:
    c = col + "_90"
    if c in centrales.columns:
        print(f"  {col:<13} mediana {centrales[c].median():.2f}"
              f"   p25 {centrales[c].quantile(.25):.2f}"
              f"   p75 {centrales[c].quantile(.75):.2f}")

print("\nCENTRALES DEL MALLORCA")

mallorca_cb = centrales[centrales["equipo_clave"] == "mallorca"]

print(mallorca_cb[["jugador", "minutes"] + columnas_90]
      .sort_values("minutes", ascending=False)
      .to_string(index=False))

print("\nPERCENTIL EN CADA METRICA")

for col in ACCIONES_DEF:
    c = col + "_90"
    if c in centrales.columns:
        centrales["pct_" + col] = (centrales[c].rank(pct=True) * 100).round(1)

cols_pct = ["pct_" + c for c in ACCIONES_DEF if "pct_" + c in centrales.columns]

mallorca_pct = centrales[centrales["equipo_clave"] == "mallorca"]

print(mallorca_pct[["jugador"] + cols_pct].to_string(index=False))

jugadores cruzados: 454 de 637
duplicados por nombre: 1

centrales comparados: 65

REFERENCIAS DE LA LIGA (centrales, minimo 600 minutos)
  Tackle        mediana 1.58   p25 1.25   p75 1.94
  Interception  mediana 1.06   p25 0.82   p75 1.25
  Clearance     mediana 5.78   p25 4.89   p75 6.52
  BlockedPass   mediana 0.47   p25 0.32   p75 0.64
  Aerial        mediana 3.36   p25 2.66   p75 4.02

CENTRALES DEL MALLORCA
       jugador  minutes  Tackle_90  Interception_90  Clearance_90  BlockedPass_90  Aerial_90
Martin Valjent     3311       1.36             0.63          6.74            0.57       2.66
Antonio Raíllo     2069       1.09             0.70          5.31            0.39       4.96
   David López      827       1.63             0.87         10.01            0.87       5.66

PERCENTIL EN CADA METRICA
       jugador  pct_Tackle  pct_Interception  pct_Clearance  pct_BlockedPass  pct_Aerial
Antonio Raíllo        15.4              16.2           40.0             40.8        92.3
Martin

In [21]:
print(liga[liga["clave"].duplicated(keep=False)]
      [["jugador", "equipo", "posicion", "minutes"]]
      .to_string(index=False))

    jugador   equipo posicion  minutes
David López Mallorca  central      827
David López   Girona  central      300


**Medianas de los 65 centrales**: entradas 1,58, intercepciones 1,06, despejes 5,78,
bloqueos de pase 0,47 y duelos aereos 3,36 por 90 minutos. Las dos primeras son los
umbrales que usa la criba de candidatos.

La ultima celda comprueba el unico nombre duplicado de la liga: hay dos David Lopez, uno
en el Mallorca (827 minutos) y otro en el Girona (300). Por eso todos los cruces se hacen
por equipo y jugador, nunca solo por nombre.

## 17. El grupo de control por altura defensiva

Comparar a los centrales del Mallorca con toda la liga tiene un problema: el equipo
defendia mas bajo que nadie, y en un bloque bajo se intercepta menos por diseno. Para
descartar esa explicacion se repite la comparacion contra los centrales de los seis
equipos que defendieron mas abajo.

*Este apartado lee `altura_defensiva_laliga_2526.csv`, que es de donde sale la altura
defensiva del trabajo y de la figura 2. Antes leia `estilo_equipos_laliga_2526.csv`, que
calcula la altura con otro conjunto de acciones y formaba un grupo ligeramente distinto.*

In [22]:
# Comparamos solo contra los centrales de equipos que defendieron igual de bajo
# Usamos el fichero de altura defensiva del notebook 02, que es el que usa el trabajo
alt = pd.read_csv("data/altura_defensiva_laliga_2526.csv")
alt["equipo_clave"] = alt["equipo"].apply(normalizar).replace(CORRECCIONES_EQUIPO)

# Los seis equipos con la linea defensiva mas retrasada de la liga
bloques_bajos = alt.nsmallest(6, "altura_defensiva")

print("EQUIPOS QUE DEFENDIERON MAS BAJO")
print(bloques_bajos[["equipo_clave", "altura_defensiva"]].to_string(index=False))

grupo = bloques_bajos["equipo_clave"].tolist()
similares = centrales[centrales["equipo_clave"].isin(grupo)].copy()

print("\ncentrales de ese grupo:", len(similares))
print("\nREFERENCIA DEL GRUPO DE BLOQUE BAJO")

for col in ACCIONES_DEF:
    c = col + "_90"
    if c in similares.columns:
        print(f"  {col:<13} mediana {similares[c].median():.2f}")

print("\nTODOS LOS CENTRALES DEL GRUPO, ORDENADOS POR INTERCEPCIONES")

print(similares[["jugador", "equipo_clave", "minutes", "Interception_90"]]
      .sort_values("Interception_90")
      .to_string(index=False))

EQUIPOS QUE DEFENDIERON MAS BAJO
equipo_clave  altura_defensiva
    mallorca             29.80
  villarreal             30.11
      girona             30.36
      oviedo             30.71
     osasuna             31.14
  celta vigo             31.25

centrales de ese grupo: 21

REFERENCIA DEL GRUPO DE BLOQUE BAJO
  Tackle        mediana 1.63
  Interception  mediana 0.96
  Clearance     mediana 6.09
  BlockedPass   mediana 0.49
  Aerial        mediana 3.02

TODOS LOS CENTRALES DEL GRUPO, ORDENADOS POR INTERCEPCIONES
          jugador equipo_clave  minutes  Interception_90
       Dani Calvo       oviedo     1688             0.53
      Enzo Boyomo      osasuna     2544             0.60
   Martin Valjent     mallorca     3311             0.63
        Yoel Lago   celta vigo      966             0.65
   Antonio Raíllo     mallorca     2069             0.70
 Carlos Domínguez   celta vigo      602             0.75
      Pau Navarro   villarreal     2034             0.80
       Rafa Marín   vil

**El grupo de control son 21 centrales y su mediana de intercepciones es 0,96, frente a
1,06 de toda la liga: solo un 9 % menos.** Defender bajo no explica la carencia.

Es el mismo grupo contra el que se dibuja la figura 6, donde los tres centrales habituales
del Mallorca quedan en los percentiles 10, 19 y 38.

## 18. El fichero maestro de metricas

Aqui se construye `metricas_jugadores_laliga_2526.csv`, el fichero que usan los notebooks
04, 05 y 06: la posicion del fichero propio, los minutos y las metricas ofensivas de
Understat, las acciones defensivas de los eventos Opta y todo convertido a valores por 90
minutos.

El cruce por nombre exacto deja 470 jugadores. Las ultimas celdas recuperan once mas por
coincidencia parcial dentro del mismo equipo, exigiendo dos palabras en comun y descartando
los empates, y comprueban quien se queda fuera.

In [23]:
# Subimos a la carpeta raiz del proyecto para que las rutas relativas funcionen
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd
from unidecode import unidecode

eventos = pd.read_parquet("data/eventos_laliga_2526.parquet")
jugadores = pd.read_csv("data/jugadores_laliga_2526.csv")
posiciones = pd.read_csv("data/posiciones_laliga_2526.csv")


def normalizar(nombre):
    n = unidecode(str(nombre)).lower().replace(".", "").replace("-", " ")
    return " ".join(n.split())


CORRECCIONES_NOMBRE = {
    "antonio raillo": "raillo",
    "justin kalumba": "justin noel kalumba",
    "manu morlanes": "manuel morlanes",
    "zito luvumbo": "zito",
}

CORRECCIONES_EQUIPO = {
    "atletico": "atletico madrid",
    "deportivo alaves": "alaves",
    "real oviedo": "oviedo",
}

# Marcamos cada tipo de accion como columna para poder sumarlas
ACCIONES = ["Tackle", "Interception", "Clearance", "BlockedPass", "Aerial"]
for accion in ACCIONES:
    eventos[accion] = eventos["type"] == accion

texto = eventos["qualifiers"].astype(str)
eventos["centros"] = (eventos["type"] == "Pass") & texto.str.contains("Cross")

toque = eventos["is_touch"].fillna(False).astype(bool)
en_area = (eventos["x"] >= 83) & eventos["y"].between(21.1, 78.9)
eventos["toques_area"] = toque & en_area

eventos["clave"] = eventos["player"].apply(normalizar)
eventos["clave"] = eventos["clave"].replace(CORRECCIONES_NOMBRE)

columnas = ACCIONES + ["centros", "toques_area"]
conteos = eventos.groupby("clave")[columnas].sum().reset_index()

print("jugadores con eventos:", len(conteos))

jugadores con eventos: 602


In [24]:
# Metricas ofensivas y minutos, que vienen de Understat
ofensivo = jugadores[["player", "team", "minutes", "goals", "xg",
                      "assists", "xa", "shots", "key_passes",
                      "xg_chain", "xg_buildup"]].copy()
ofensivo["clave"] = ofensivo["player"].apply(normalizar)
ofensivo["clave"] = ofensivo["clave"].replace(CORRECCIONES_NOMBRE)
ofensivo["equipo_clave"] = ofensivo["team"].apply(normalizar)
ofensivo["equipo_clave"] = ofensivo["equipo_clave"].replace(CORRECCIONES_EQUIPO)

# Posicion, que viene de mi fichero propio
pos = posiciones[["jugador", "equipo", "posicion"]].copy()
pos["clave"] = pos["jugador"].apply(normalizar)
pos["clave"] = pos["clave"].replace(CORRECCIONES_NOMBRE)
pos["equipo_clave"] = pos["equipo"].apply(normalizar)
pos["equipo_clave"] = pos["equipo_clave"].replace(CORRECCIONES_EQUIPO)

metricas = (pos
            .merge(ofensivo, on=["clave", "equipo_clave"], how="inner")
            .merge(conteos, on="clave", how="left"))

print("jugadores cruzados:", len(metricas), "de", len(pos))

# Pasamos todo a valores por 90 minutos
BRUTAS = ["goals", "xg", "assists", "xa", "shots", "key_passes",
          "xg_chain", "xg_buildup"] + columnas

for col in BRUTAS:
    metricas[col + "_90"] = metricas[col] / metricas["minutes"] * 90

metricas["dif_goles_xg"] = metricas["goals"] - metricas["xg"]

metricas.to_csv("data/metricas_jugadores_laliga_2526.csv", index=False)
print("guardado en data/metricas_jugadores_laliga_2526.csv")

jugadores cruzados: 470 de 637
guardado en data/metricas_jugadores_laliga_2526.csv


In [25]:
for nombre in ["Vedat Muriqi", "Sergi Darder", "Pablo Maffeo", "Pablo Torre"]:
    clave = normalizar(nombre)
    fila = metricas[metricas["clave"] == clave]
    if len(fila) == 0:
        print("NO ENCONTRADO:", nombre)
    else:
        f = fila.iloc[0]
        print(f"{nombre}: {f['posicion']}, {int(f['minutes'])} min")

Vedat Muriqi: delantero, 3150 min
Sergi Darder: mediocentro, 2715 min
Pablo Maffeo: lateral, 2690 min
Pablo Torre: mediapunta, 1614 min


In [26]:
# Para los que no cruzan por nombre exacto, probamos coincidencia parcial
# dentro del mismo equipo
sin_cruzar = pos[~pos["clave"].isin(metricas["clave"])].copy()
print("jugadores sin cruzar:", len(sin_cruzar))

rescates = []

for _, fila in sin_cruzar.iterrows():
    tokens = set(fila["clave"].split())
    candidatos = ofensivo[ofensivo["equipo_clave"] == fila["equipo_clave"]]

    mejor, mejor_n = None, 0
    empate = False
    for _, cand in candidatos.iterrows():
        comunes = len(tokens & set(cand["clave"].split()))
        if comunes > mejor_n:
            mejor, mejor_n, empate = cand["clave"], comunes, False
        elif comunes == mejor_n and comunes > 0:
            empate = True

    # Solo aceptamos si comparten dos palabras y no hay empate
    if mejor is not None and mejor_n >= 2 and not empate:
        rescates.append((fila["jugador"], fila["clave"], mejor))

print("recuperados:", len(rescates))
print()
for original, clave_vieja, clave_nueva in rescates[:40]:
    print(f"  {original:<28} -> {clave_nueva}")

jugadores sin cruzar: 166


recuperados: 11

  Pape Gueye                   -> pape alassane gueye
  Pau Cabanes                  -> pau cabanes de la torre
  Bryan Gil                    -> bryan gil salvatierra
  Jon Pacheco                  -> jon pacheco dozagarat
  Pablo Barrios                -> pablo barrios rivas
  Miguel Rubio                 -> miguel angel rubio
  Urko González de Zárate      -> urko gonzalez
  Jon Olasagasti               -> jon ander olasagasti
  Karl Etta Eyong              -> etta eyong
  Kylian Mbappé                -> kylian mbappe lottin
  Arnaut Danjuma               -> arnaut danjuma groeneveld


In [27]:
# Aplicamos los once emparejamientos recuperados
RESCATES = {vieja: nueva for _, vieja, nueva in rescates}
print("rescates aplicados:", len(RESCATES))

pos["clave"] = pos["clave"].replace(RESCATES)

metricas = (pos
            .merge(ofensivo, on=["clave", "equipo_clave"], how="inner")
            .merge(conteos, on="clave", how="left"))

for col in BRUTAS:
    metricas[col + "_90"] = metricas[col] / metricas["minutes"] * 90

metricas["dif_goles_xg"] = metricas["goals"] - metricas["xg"]

metricas.to_csv("data/metricas_jugadores_laliga_2526.csv", index=False)
print("jugadores cruzados:", len(metricas), "de", len(pos))

rescates aplicados: 11
jugadores cruzados: 481 de 637


In [28]:
# ¿Se queda fuera algun jugador del Mallorca o algun central?
fuera = pos[~pos["clave"].isin(metricas["clave"])]

print("del Mallorca sin cruzar:")
print(fuera[fuera["equipo_clave"] == "mallorca"]
      [["jugador", "posicion"]].to_string(index=False))

print("\ncentrales sin cruzar:", int((fuera["posicion"] == "central").sum()))
print("por posicion:")
print(fuera["posicion"].value_counts().to_string())

del Mallorca sin cruzar:
      jugador    posicion
Iliesse Salhi sin_definir
  Leo Sánchez sin_definir
    Cesc Riba sin_definir
Jandro García sin_definir
   Cyle Larin   delantero

centrales sin cruzar: 25
por posicion:
posicion
sin_definir    39
lateral        26
central        25
delantero      24
extremo        22
mediocentro    13
mediapunta      5
pivote          1


**481 jugadores de los 637 del fichero de posiciones.** Los que quedan fuera son en su
mayoria jugadores sin posicion asignada o con muy pocos minutos. Del Mallorca solo se
pierden cuatro canteranos sin posicion definida y Cyle Larin.

Quedan 25 centrales sin cruzar, todos de otros equipos. El grupo de referencia del
apartado 16 se construye por separado a partir de los eventos, asi que no depende de este
cruce.

---
## Conclusiones del analisis individual

1. **Los centrales son el punto debil.** Entre los 65 centrales de LaLiga con 600 minutos
   o mas, Valjent queda en el percentil 9 (0,63 intercepciones por 90), Raillo en el 12
   (0,70) y David Lopez en el 31 (0,87). Contra el grupo de control de 21 centrales de
   equipos que defendian igual de bajo, los percentiles son 10, 19 y 38.
2. **Defender bajo no lo explica**: la mediana del grupo de control es 0,96 frente a 1,06
   de toda la liga, un 9 % menos.
3. **No habia recambio interno**: los cuatro centrales de la plantilla estan por debajo de
   la mediana en anticipacion (0,87 / 0,70 / 0,63 / 0,41 frente a 1,06).
4. **Las bandas no eran el problema.** Maffeo esta por encima de la mediana en
   intercepciones y en entradas; Mojica es un lateral ofensivo y jugaba para eso.
5. Las metricas defensivas altas de delanteros y mediapuntas son **sintoma del bloque
   bajo**, no fortalezas individuales.

El perfil de refuerzo que sale de aqui: **central que anticipe por encima de la mediana
sin perder juego aereo**. La busqueda esta en el notebook 04.